# SegFormer-B0 on DRIVE (Binary Segmentation)

- Transfer learning with pretrained SegFormer-B0
- AdamW optimizer
- Early stopping on validation Dice or IoU
- Saves best model to best_segformer_b0.pth

In [ ]:
!pip -q install transformers timm albumentations opencv-python tqdm pandas matplotlib

In [ ]:
import os
from pathlib import Path
import random
import zipfile

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
DATA_DIR = Path("/kaggle/input/drive-processed-dataset")
ZIP_PATH = DATA_DIR / "DRIVE_processed_dataset.zip"
WORK_DIR = Path("/kaggle/working")
EXTRACT_DIR = WORK_DIR / "DRIVE_processed_dataset"

IMG_SIZE = 512
BATCH_SIZE = 4
NUM_EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 10
BEST_METRIC = "dice"
AMP = torch.cuda.is_available()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BEST_MODEL_PATH = WORK_DIR / "best_segformer_b0.pth"

In [ ]:
if ZIP_PATH.exists() and not EXTRACT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)

if (EXTRACT_DIR / "train").exists():
    DATA_ROOT = EXTRACT_DIR
else:
    DATA_ROOT = DATA_DIR

print("DATA_ROOT:", DATA_ROOT)
print("train files:", len(list((DATA_ROOT / "train").glob("*.pt"))))

In [ ]:
class DriveProcessedDataset(Dataset):
    def __init__(self, root_dir: Path, split: str, img_size: int | None = None):
        self.root_dir = Path(root_dir)
        self.split = split
        self.files = sorted((self.root_dir / split).glob("*.pt"))
        self.img_size = img_size

    def __len__(self) -> int:
        return len(self.files)

    def __getitem__(self, idx: int):
        sample = torch.load(self.files[idx], map_location="cpu")
        image = sample.get("image")
        mask = sample.get("mask") or sample.get("manual_1") or sample.get("manual")

        if not torch.is_tensor(image):
            image = torch.tensor(image)
        if image.dim() == 3 and image.shape[0] != 3:
            image = image.permute(2, 0, 1)
        image = image.float()
        if image.max() > 1.0:
            image = image / 255.0

        if not torch.is_tensor(mask):
            mask = torch.tensor(mask)
        mask = mask.float()
        if mask.dim() == 2:
            mask = mask.unsqueeze(0)
        if mask.max() > 1.0:
            mask = mask / 255.0

        if self.img_size is not None:
            target_size = (self.img_size, self.img_size)
            if image.shape[-2:] != target_size:
                image = F.interpolate(
                    image.unsqueeze(0),
                    size=target_size,
                    mode="bilinear",
                    align_corners=False,
                ).squeeze(0)
            if mask.shape[-2:] != target_size:
                mask = F.interpolate(
                    mask.unsqueeze(0),
                    size=target_size,
                    mode="nearest",
                ).squeeze(0)

        return image, mask

train_ds = DriveProcessedDataset(DATA_ROOT, "train", img_size=IMG_SIZE)
val_ds = DriveProcessedDataset(DATA_ROOT, "val", img_size=IMG_SIZE)
test_ds = DriveProcessedDataset(DATA_ROOT, "test", img_size=IMG_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print("train/val/test:", len(train_ds), len(val_ds), len(test_ds))

In [ ]:
try:
    from src.models.segformer import SegFormerB0
except Exception:
    from transformers import SegformerConfig, SegformerForSemanticSegmentation
    import torch.nn as nn

    class SegFormerB0(nn.Module):
        def __init__(
            self,
            pretrained: bool = True,
            pretrained_model_name: str = "nvidia/segformer-b0-finetuned-ade-512-512",
            num_labels: int = 1,
            ignore_mismatched_sizes: bool = True,
        ) -> None:
            super().__init__()
            config = SegformerConfig.from_pretrained(pretrained_model_name)
            config.num_labels = num_labels

            if pretrained:
                self.model = SegformerForSemanticSegmentation.from_pretrained(
                    pretrained_model_name,
                    config=config,
                    ignore_mismatched_sizes=ignore_mismatched_sizes,
                )
            else:
                self.model = SegformerForSemanticSegmentation(config)

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            outputs = self.model(pixel_values=x)
            logits = outputs.logits
            if logits.shape[-2:] != x.shape[-2:]:
                logits = F.interpolate(
                    logits, size=x.shape[-2:], mode="bilinear", align_corners=False
                )
            return logits

model = SegFormerB0(pretrained=True).to(DEVICE)

images, masks = next(iter(train_loader))
with torch.no_grad():
    logits = model(images.to(DEVICE))
print("input:", images.shape, "logits:", logits.shape)

In [ ]:
bce_loss = torch.nn.BCEWithLogitsLoss()

def dice_coef(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    intersection = (probs * targets).sum(dim=(2, 3))
    denom = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3))
    dice = (2.0 * intersection + eps) / (denom + eps)
    return dice.mean()

def iou_score(logits: torch.Tensor, targets: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    probs = torch.sigmoid(logits)
    intersection = (probs * targets).sum(dim=(2, 3))
    union = probs.sum(dim=(2, 3)) + targets.sum(dim=(2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()

def bce_dice_loss(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return bce_loss(logits, targets) + (1.0 - dice_coef(logits, targets))


class EarlyStopping:
    def __init__(self, patience: int = 10, min_delta: float = 0.0, mode: str = "max") -> None:
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best_score = None
        self.counter = 0

    def step(self, score: float) -> bool:
        if self.best_score is None:
            self.best_score = score
            return False
        if self.mode == "max":
            improved = score > (self.best_score + self.min_delta)
        else:
            improved = score < (self.best_score - self.min_delta)
        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
        return self.counter >= self.patience


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=AMP):
            logits = model(images)
            loss = bce_dice_loss(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    total_iou = 0.0
    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = bce_dice_loss(logits, masks)
        dice = dice_coef(logits, masks)
        iou = iou_score(logits, masks)
        total_loss += loss.item() * images.size(0)
        total_dice += dice.item() * images.size(0)
        total_iou += iou.item() * images.size(0)
    n = len(loader.dataset)
    return total_loss / n, total_dice / n, total_iou / n


def train_model(model, train_loader, val_loader):
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP)
    early = EarlyStopping(patience=PATIENCE, min_delta=1e-4, mode="max")
    history = []
    best_score = -1.0
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
        val_loss, val_dice, val_iou = evaluate(model, val_loader)
        metric = val_dice if BEST_METRIC == "dice" else val_iou
        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_dice": val_dice,
            "val_iou": val_iou,
        })
        print(
            f"Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} | ",
            f"dice {val_dice:.4f} | iou {val_iou:.4f}",
        )
        if metric > best_score + 1e-4:
            best_score = metric
            torch.save(model.state_dict(), BEST_MODEL_PATH)
            print(f"Saved best model to {BEST_MODEL_PATH}")
        if early.step(metric):
            print("Early stopping triggered.")
            break
    return history


history = train_model(model, train_loader, val_loader)
history_df = pd.DataFrame(history)
history_df.tail()